# AutoDL DenseNet121 / CheXNet-style Training Notebook

This notebook is for running the DenseNet121 baseline on AutoDL or another CUDA cloud host. It keeps the training pipeline, commands, and resulting metrics in one place.

Recommended workflow:

1. Run environment and CUDA checks.
2. Verify NIH data paths.
3. Run DenseNet121 smoke training.
4. Inspect smoke metrics.
5. Run full DenseNet121 baseline training.
6. Inspect final test metrics and per-class metrics.

The full baseline uses `configs/cnn_densenet121_baseline.yaml` and automatically evaluates the best validation-AUROC checkpoint on the test split.

## 0. Notebook Settings

This notebook lives under `visiontrans_diagnostics/notebooks/`, but all commands should run from the git repo root.

The next cell automatically detects the repo root if Jupyter starts in either `visiontrans_diagnostics` or `visiontrans_diagnostics/notebooks`. If AutoDL opens Jupyter somewhere else, set `REPO_ROOT` manually.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

# If needed on AutoDL, replace this with the absolute path to visiontrans_diagnostics.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "run_experiment.py").exists() and (REPO_ROOT.parent / "src" / "run_experiment.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "src" / "run_experiment.py").exists():
    raise RuntimeError(f"REPO_ROOT does not look like visiontrans_diagnostics: {REPO_ROOT}")

os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)
print("Python:", sys.executable)

In [ ]:
def run_command(cmd, cwd=REPO_ROOT, env=None):
    """Run a shell command and stream output into the notebook."""
    print("$", " ".join(str(part) for part in cmd))
    start = time.time()
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    code = process.wait()
    elapsed = time.time() - start
    print(f"\n[exit_code={code}] elapsed={elapsed/60:.2f} min")
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")
    return "".join(lines)

## 1. Environment and CUDA Check

On AutoDL, use the CUDA environment from `environment_cuda.yml` if you created it. This baseline expects PyTorch with CUDA available for full training.

In [ ]:
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda version:", torch.version.cuda)
    print("gpu count:", torch.cuda.device_count())
    print("gpu name:", torch.cuda.get_device_name(0))
    print("gpu capability:", torch.cuda.get_device_capability(0))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Selected DEVICE:", DEVICE)

In [ ]:
# Optional: show NVIDIA driver / GPU utilization on AutoDL.
try:
    run_command(["nvidia-smi"])
except Exception as exc:
    print("nvidia-smi check skipped or failed:", exc)

## 2. Data Path Check

Expected layout on AutoDL. The `data` directory should be parallel to the git repo, not inside it:

```text
parent_dir/
  visiontrans_diagnostics/
  data/
    raw/images-224/images-224/*.png
    annotations/Data_Entry_2017.csv
    annotations/train_val_list.txt
    annotations/test_list.txt
```

If manifests were copied from Mac, they may contain Mac absolute image paths. The full training command below uses `--force-manifests` to rebuild Linux paths.

In [ ]:
from pathlib import Path

paths_to_check = [
    Path("../data/raw"),
    Path("../data/annotations/Data_Entry_2017.csv"),
    Path("../data/annotations/train_val_list.txt"),
    Path("../data/annotations/test_list.txt"),
]
for path in paths_to_check:
    print(f"{str(path):45s}", "OK" if path.exists() else "MISSING")

num_images = sum(1 for _ in Path("../data/raw").rglob("*.png")) if Path("../data/raw").exists() else 0
print("png images under ../data/raw:", num_images)

## 3. Inspect DenseNet121 Configs

Smoke config uses a small sampled manifest and `pretrained: false` for speed. Full baseline config uses ImageNet pretrained DenseNet121.

In [ ]:
import yaml

for config_path in ["configs/cnn_densenet121_smoke.yaml", "configs/cnn_densenet121_baseline.yaml"]:
    print("\n===", config_path, "===")
    print(Path(config_path).read_text())

## 4. Smoke Training

Run this first. It should finish quickly and produce a checkpoint plus smoke test metrics.

Smoke output files:

```text
artifacts/models/cnn_densenet121_smoke_best.pt
artifacts/metrics/cnn_densenet121_smoke_history.csv
artifacts/metrics/cnn_densenet121_smoke_val_auc.json
artifacts/metrics/cnn_densenet121_smoke_test_metrics.json
artifacts/metrics/cnn_densenet121_smoke_test_per_class_metrics.csv
```

In [ ]:
SMOKE_CONFIG = "configs/cnn_densenet121_smoke.yaml"

run_command([
    sys.executable,
    "src/run_experiment.py",
    "--config", SMOKE_CONFIG,
    "--device", DEVICE,
    "--no-mlflow",
])

## 5. Smoke Results

In [ ]:
import pandas as pd

smoke_metrics_path = Path("artifacts/metrics/cnn_densenet121_smoke_test_metrics.json")
smoke_per_class_path = Path("artifacts/metrics/cnn_densenet121_smoke_test_per_class_metrics.csv")

if smoke_metrics_path.exists():
    smoke_metrics = json.loads(smoke_metrics_path.read_text())
    print("Smoke test summary:")
    for key in ["loss", "mean_auroc", "mean_pr_auc", "macro_f1", "micro_f1", "macro_precision", "macro_recall"]:
        print(f"{key:20s}: {smoke_metrics.get(key)}")
else:
    print("Missing", smoke_metrics_path)

if smoke_per_class_path.exists():
    display(pd.read_csv(smoke_per_class_path))

## 6. Full DenseNet121 Baseline Training

Run this after smoke training passes. On AutoDL/4090, this is the main run.

The command uses `--force-manifests` so Linux manifests are rebuilt with correct AutoDL image paths.

The training script prints live progress inside the notebook output. For the full baseline, `training.progress_interval: 50` means you will see updates like `epoch=3/30 batch=250/1400 (17.9%) train_loss_avg=...` every 50 batches, plus an epoch summary after validation.

Expected final outputs:

```text
artifacts/models/densenet121_chexnet_style_baseline_best.pt
artifacts/metrics/densenet121_chexnet_style_baseline_history.csv
artifacts/metrics/densenet121_chexnet_style_baseline_val_auc.json
artifacts/metrics/densenet121_chexnet_style_baseline_criterion.json
artifacts/metrics/densenet121_chexnet_style_baseline_test_metrics.json
artifacts/metrics/densenet121_chexnet_style_baseline_test_per_class_metrics.csv
```

In [ ]:
BASELINE_CONFIG = "configs/cnn_densenet121_baseline.yaml"

# Set RUN_FULL_BASELINE = True only after smoke training succeeds.
RUN_FULL_BASELINE = False

if RUN_FULL_BASELINE:
    run_command([
        sys.executable,
        "src/run_experiment.py",
        "--config", BASELINE_CONFIG,
        "--device", "cuda",
        "--force-manifests",
    ])
else:
    print("Full baseline is disabled. Set RUN_FULL_BASELINE = True when ready.")

## 7. Full Baseline Results

Run after the full baseline training cell finishes.

In [ ]:
baseline_metrics_path = Path("artifacts/metrics/densenet121_chexnet_style_baseline_test_metrics.json")
baseline_per_class_path = Path("artifacts/metrics/densenet121_chexnet_style_baseline_test_per_class_metrics.csv")
baseline_history_path = Path("artifacts/metrics/densenet121_chexnet_style_baseline_history.csv")

if baseline_metrics_path.exists():
    baseline_metrics = json.loads(baseline_metrics_path.read_text())
    print("DenseNet121 baseline test summary:")
    for key in ["loss", "mean_auroc", "mean_pr_auc", "macro_f1", "micro_f1", "macro_precision", "macro_recall", "checkpoint_epoch", "checkpoint_val_mean_auc"]:
        print(f"{key:25s}: {baseline_metrics.get(key)}")
else:
    print("Missing", baseline_metrics_path)

if baseline_per_class_path.exists():
    display(pd.read_csv(baseline_per_class_path).sort_values("auroc", ascending=False))

if baseline_history_path.exists():
    display(pd.read_csv(baseline_history_path).tail())

## 8. Re-evaluate Best Checkpoint Manually

Use this if you need to re-run test evaluation without retraining.

In [ ]:
RUN_MANUAL_EVAL = False

if RUN_MANUAL_EVAL:
    run_command([
        sys.executable,
        "src/evaluate.py",
        "--config", BASELINE_CONFIG,
        "--checkpoint", "artifacts/models/densenet121_chexnet_style_baseline_best.pt",
        "--split", "test",
        "--device", "cuda",
    ])
else:
    print("Manual evaluation is disabled. Set RUN_MANUAL_EVAL = True if needed.")

## 9. Notes and Run Log

Use this section to record AutoDL instance details, run dates, observations, or issues.

- AutoDL instance:
- GPU:
- CUDA/PyTorch:
- Dataset location:
- Smoke run status:
- Full baseline status:
- Best checkpoint:
- Main result:
- Notes: